# Evaluation Comparison — Fine-Tuned Model vs. Baseline

**Notebook purpose:** Evaluate the fine-tuned Khmer Whisper model (LoRA adapter,
3 epochs, 68,059 additional training samples) against the untouched baseline
checkpoint, on the same held-out evaluation sets used in `EVAL-02`. This produces
the "Improvement Delta" — the core evidence of whether fine-tuning actually worked.

**Baseline reference (from EVAL-02):**
- CER: 41.10% (FLEURS Khmer test split, 776 samples)
- WER: 457.15% (unreliable — Khmer word-spacing artifacts, documented in EVAL-02)

## Related task
`[EVAL-03] Evaluate Fine-Tuned Model vs. Baseline` (see Trello board)

Library

In [ ]:
import os
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel
import librosa

In [ ]:
import librosa
from datasets import load_dataset
import soundfile as sf

In [ ]:
import json
from getpass import getpass
import os

Mount Drive & set paths:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/khmer-stt-bot'
FINAL_ADAPTER_DIR = f"{DRIVE_PROJECT_DIR}/models/checkpoints/final_adapter"
EVAL_DIR = f"{DRIVE_PROJECT_DIR}/data/eval/fleurs_km"   # <-- now lives in Drive
os.makedirs(EVAL_DIR, exist_ok=True) # if running fresh, re-download per DATA-03, or point at Drive copy if saved there

In [ ]:
PROCESSED_DIR = f"{DRIVE_PROJECT_DIR}/data/processed"
CKPT_DIR      = f"{DRIVE_PROJECT_DIR}/checkpoints"
RESULTS_DIR   = f"{DRIVE_PROJECT_DIR}/results"
REAL_WORLD_DIR= f"{DRIVE_PROJECT_DIR}/data/eval/real_world"

In [ ]:

from safetensors.torch import load_file

check_state = load_file(f"{FINAL_ADAPTER_DIR}/adapter_model.safetensors")
lora_b_keys = [k for k in check_state if "lora_B" in k]
total_b_norm = sum(check_state[k].norm().item() for k in lora_b_keys)
print(f"lora_B norm in final_adapter now: {total_b_norm:.6f}")

In [ ]:
import os
print(sorted(os.listdir(FINAL_ADAPTER_DIR)))

Install dependencies:

In [ ]:
!pip install -q transformers peft librosa jiwer soundfile

In [ ]:
!pip install -U torchao

**Load the fine-tuned model:**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
base_model = WhisperForConditionalGeneration.from_pretrained("seanghay/whisper-small-khmer-v2")
finetuned_model = PeftModel.from_pretrained(base_model, FINAL_ADAPTER_DIR)
finetuned_model = finetuned_model.to(device)
finetuned_model.eval()

In [ ]:
processor = WhisperProcessor.from_pretrained(FINAL_ADAPTER_DIR)

# Same forced_decoder_ids fix from DATA-02 — required for this checkpoint's config format
finetuned_model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="km", task="transcribe"
)

print("Fine-tuned model loaded and ready")

**Transcribe function (same as baseline, for a fair comparison):**

In [ ]:
prompt_length = len(finetuned_model.generation_config.forced_decoder_ids)
MAX_TOTAL_TOKENS = 448
SAFETY_MARGIN = 10

def transcribe(audio_path, sample_rate=16000):
    audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
    duration_sec = len(audio) / sample_rate

    dynamic_max_tokens = min(
        int(duration_sec * 40) + 80,
        MAX_TOTAL_TOKENS - prompt_length - SAFETY_MARGIN
    )

    inputs = processor(audio, sampling_rate=sample_rate, return_tensors="pt").to(device)

    with torch.no_grad():
        predicted_ids = finetuned_model.generate(
            inputs["input_features"],
            max_new_tokens=dynamic_max_tokens,
        )

    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text

**Load ground truth (same FLEURS set as baseline):**

In [ ]:
fleurs_km = load_dataset("google/fleurs", "km_kh", split="test")

for i, sample in enumerate(fleurs_km):
    filename = f"fleurs_{i:04d}.wav"
    filepath = os.path.join(EVAL_DIR, filename)
    if os.path.exists(filepath):
        continue  # resume-safe, skip files already saved
    sf.write(filepath, sample["audio"]["array"], sample["audio"]["sampling_rate"])

print(f"Audio restored to {EVAL_DIR}")

In [ ]:
transcript_lines = [f"{fn}\t{ground_truth[fn]}" for fn in sorted(ground_truth.keys())]
with open(f"{EVAL_DIR}/transcripts.tsv", "w", encoding="utf-8") as f:
    f.write("\n".join(transcript_lines))

In [ ]:
import csv

metadata_path = f"{EVAL_DIR}/transcripts.tsv"
ground_truth = {}

with open(metadata_path, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
    for row in reader:
        if len(row) < 2 or not row[1].strip():
            continue
        ground_truth[row[0]] = row[1]

print(f"Loaded {len(ground_truth)} ground-truth transcripts")

**Run inference on the fine-tuned model (resume-safe, learned from EVAL-02's disconnect scare):**

In [ ]:
os.makedirs(f"{DRIVE_PROJECT_DIR}/results", exist_ok=True)
RESULTS_PATH = f"{DRIVE_PROJECT_DIR}/results/finetuned_predictions.json"

In [ ]:
print(os.path.exists(RESULTS_PATH))

In [ ]:
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as f:
        predictions = json.load(f)
    print(f"Resuming — {len(predictions)} files already done")
else:
    predictions = {}

remaining = [f for f in sorted(ground_truth.keys()) if f not in predictions]
print(f"{len(remaining)} files left to transcribe")

for i, filename in enumerate(remaining):
    filepath = os.path.join(EVAL_DIR, filename)
    predictions[filename] = transcribe(filepath)

    if i % 20 == 0:
        with open(RESULTS_PATH, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        print(f"  progress: {len(predictions)}/{len(ground_truth)}")

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"Done. {len(predictions)} predictions saved.")

**Compute CER/WER using tested metrics.py utility:**

In [ ]:
"""
CER/WER scoring utilities for the Khmer STT project.

CER (Character Error Rate) is the primary metric for this project — Khmer script
has no spaces between words, so word-level boundaries used by WER are ambiguous.
WER is still computed and reported for comparability with prior published Khmer
ASR work (e.g. whisper-small-khmer-v2's reported WER).
"""

from jiwer import cer, wer


def compute_cer(reference: str, hypothesis: str) -> float:
    """
    Character Error Rate between a single reference (ground truth) and
    hypothesis (model prediction) string. Returns a float, e.g. 0.42 = 42% error.
    """
    return cer(reference, hypothesis)


def compute_wer(reference: str, hypothesis: str) -> float:
    """
    Word Error Rate between a single reference and hypothesis string.
    Returns a float, e.g. 0.62 = 62% error.
    """
    return wer(reference, hypothesis)


def compute_batch_cer(references: list[str], hypotheses: list[str]) -> float:
    """
    CER computed across a full list of reference/hypothesis pairs at once.
    This is the aggregate/overall CER — not an average of per-file CERs, since
    jiwer pools all edit distances and lengths together for the true overall rate.
    """
    return cer(references, hypotheses)


def compute_batch_wer(references: list[str], hypotheses: list[str]) -> float:
    """
    WER computed across a full list of reference/hypothesis pairs at once.
    """
    return wer(references, hypotheses)

In [ ]:
import sys
sys.path.insert(0, "src")

filenames = sorted(ground_truth.keys())
refs = [ground_truth[f] for f in filenames]
hyps = [predictions[f] for f in filenames]

finetuned_cer = compute_batch_cer(refs, hyps)
finetuned_wer = compute_batch_wer(refs, hyps)

print(f"Fine-tuned CER: {finetuned_cer:.4f} ({finetuned_cer*100:.2f}%)")
print(f"Fine-tuned WER: {finetuned_wer:.4f} ({finetuned_wer*100:.2f}%)")

**Compute the Improvement Delta:**

In [ ]:
BASELINE_CER = 0.4110  # from EVAL-02

improvement_delta = BASELINE_CER - finetuned_cer
relative_improvement = (improvement_delta / BASELINE_CER) * 100

print(f"Baseline CER:     {BASELINE_CER*100:.2f}%")
print(f"Fine-tuned CER:   {finetuned_cer*100:.2f}%")
print(f"Improvement:      {improvement_delta*100:.2f} percentage points")
print(f"Relative gain:    {relative_improvement:.1f}%")

**Save example transcriptions for the thesis (a few good, a few still-imperfect):**

In [ ]:
import random

sample_files = random.sample(filenames, 5)
examples = []
for f in sample_files:
    examples.append({
        "file": f,
        "reference": ground_truth[f],
        "baseline_note": "(see docs/baseline_results.md for baseline prediction)",
        "finetuned_prediction": predictions[f]
    })

for ex in examples:
    print(f"File: {ex['file']}")
    print(f"  Reference:  {ex['reference']}")
    print(f"  Fine-tuned: {ex['finetuned_prediction']}")
    print()

## Step: Real-World Evaluation — Baseline vs. Fine-Tuned

Evaluate both the untouched baseline checkpoint (`whisper-small-khmer-v2`) and this
project's fine-tuned model on the same 38-recording real-world holdout set
(`data/eval/real_world/`) — spontaneous, phone-mic-quality Khmer speech, never used
in training.


## Related task
`[EVAL-03]` (Evaluate Fine-Tuned vs. Baseline), `[EVAL-06]` (Full Pipeline Comparison)

**Load real-world ground truth**

In [ ]:
import csv

real_world_truth = {}
with open(f"{REAL_WORLD_DIR}/transcripts.tsv", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
    for row in reader:
        if len(row) < 2 or not row[1].strip():
            continue
        real_world_truth[row[0]] = row[1]

print(f"Loaded {len(real_world_truth)} real-world ground-truth transcripts")

**Load the baseline mode**

In [ ]:
from transformers import WhisperForConditionalGeneration

baseline_model = WhisperForConditionalGeneration.from_pretrained("seanghay/whisper-small-khmer-v2")
baseline_model = baseline_model.to(device)
baseline_model.eval()
baseline_model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="km", task="transcribe"
)

print("Baseline model loaded")

**Transcribe function for the baseline model:**

In [ ]:
def transcribe_baseline(audio_path, sample_rate=16000):
    audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
    duration_sec = len(audio) / sample_rate
    dynamic_max_tokens = min(int(duration_sec * 40) + 80, 438)

    inputs = processor(audio, sampling_rate=sample_rate, return_tensors="pt").to(device)
    with torch.no_grad():
        predicted_ids = baseline_model.generate(
            inputs["input_features"],
            max_new_tokens=dynamic_max_tokens,
        )
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

**Run BOTH models on the same 38 real-world files:**

In [ ]:
rw_filenames = sorted(real_world_truth.keys())
rw_refs = [real_world_truth[f] for f in rw_filenames]

baseline_rw_predictions = {}
finetuned_rw_predictions = {}

for filename in rw_filenames:
    filepath = os.path.join(REAL_WORLD_DIR, filename)
    baseline_rw_predictions[filename] = transcribe_baseline(filepath)
    finetuned_rw_predictions[filename] = transcribe(filepath)  # your existing fine-tuned transcribe()

print(f"Transcribed {len(rw_filenames)} files with both models")

**Compute CER/WER for both**

In [ ]:
baseline_rw_hyps = [baseline_rw_predictions[f] for f in rw_filenames]
finetuned_rw_hyps = [finetuned_rw_predictions[f] for f in rw_filenames]

baseline_rw_cer = compute_batch_cer(rw_refs, baseline_rw_hyps)
baseline_rw_wer = compute_batch_wer(rw_refs, baseline_rw_hyps)

finetuned_rw_cer = compute_batch_cer(rw_refs, finetuned_rw_hyps)
finetuned_rw_wer = compute_batch_wer(rw_refs, finetuned_rw_hyps)

print(f"Baseline   Real-World CER: {baseline_rw_cer*100:.2f}%  |  WER: {baseline_rw_wer*100:.2f}%")
print(f"Fine-tuned Real-World CER: {finetuned_rw_cer*100:.2f}%  |  WER: {finetuned_rw_wer*100:.2f}%")

**Full 2×2 comparison matrix, all four numbers together:**

In [ ]:
print("=" * 55)
print(f"{'':12} {'FLEURS (n=776)':>18} {'Real-World (n=38)':>20}")
print("=" * 55)
print(f"{'Baseline':12} {'41.10%':>18} {baseline_rw_cer*100:>19.2f}%")
print(f"{'Fine-tuned':12} {finetuned_cer*100:>17.2f}% {finetuned_rw_cer*100:>19.2f}%")
print("=" * 55)

fleurs_delta = 41.10 - (finetuned_cer * 100)
rw_delta = (baseline_rw_cer - finetuned_rw_cer) * 100
print(f"\nFLEURS Improvement:     {fleurs_delta:.2f} points")
print(f"Real-World Improvement: {rw_delta:.2f} points")

**Save a few side-by-side examples**

In [ ]:
import random

sample_files = random.sample(rw_filenames, 3)
for f in sample_files:
    print(f"File: {f}")
    print(f"  Reference:  {real_world_truth[f]}")
    print(f"  Baseline:   {baseline_rw_predictions[f]}")
    print(f"  Fine-tuned: {finetuned_rw_predictions[f]}")
    print()

**Results write-up**

## Results Summary

| Metric | Baseline (`v2`) | Fine-Tuned (this project) | Improvement |
|---|---|---|---|
| FLUERs (CER) | 41.10% | **32.27%** | **8.83%** |
| Real World (CER) | 57.89% | **50.47%** | **7.42%**  |  

Fine-tuned on 68,059 additional samples (DDD-Cambodia raw + augmented) using LoRA,
3 epochs, r=16/alpha=32, over ~6 days spanning Colab and Kaggle sessions.

**Next steps:** log this into `docs/eval_results.md` (`EVAL-03`), then move to real-world
holdout evaluation and LLM correction (`EVAL-05`/`EVAL-06`).

## EVAL-05 — LLM Enhancement Delta (Gemini Correction Layer)

**Goal:** quantify how much the Gemini correction layer improves on the fine-tuned
Whisper model's raw output — the comparison this thesis calls the **LLM Enhancement
Delta**, measured as CER/WER before vs. after correction, on both eval sets.

**Model:** `gemini-3.5-flash-lite` (free tier). Originally planned around `gemini-3.8-flash`,
but that model's free tier is capped at 20 requests/day — not enough to correct ~800 eval
files on a thesis timeline, and a $30 minimum prepay made the paid tier impractical for this
project's budget. `gemini-3.5-flash-lite` gives 500 requests/day free; quality is noticeably
behind the flagship model on spot checks, but remains usable. This trade-off is documented
here and in `docs/results.md` as an honest budget/quota constraint, not a silent substitution.

**Quota constraint:** 500 requests/day is shared across both eval sets combined (771 FLEURS +
38 real-world = 809 files), so this will likely take **2 days** to fully complete. The
correction script below tracks a single shared daily counter and resumes safely across
sessions/days — see the `EVAL-03` postmortem in `docs/results.md` for why resume-safety is
handled explicitly rather than assumed.

**Inputs:**
- FLEURS raw predictions — reused from the already-verified `EVAL-03` run (`finetuned_predictions.json`)
- Real-world raw predictions — re-transcribed fresh in this notebook (not previously saved)

**Output:** CER/WER for raw fine-tuned ASR vs. fine-tuned+Gemini-corrected, per eval set,
with the Enhancement Delta in both absolute (pp) and relative (%) terms.

In [ ]:
%cd /content/drive/MyDrive/khmer-stt-bot

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")   # get one free at aistudio.google.com

In [ ]:
import json, os, datetime
from src.correction import correct_text, GeminiCorrectionError

MODEL_NAME = "gemini-3.5-flash-lite"
DAILY_QUOTA = 500
QUOTA_LOG_PATH = f"{RESULTS_DIR}/correction_quota_log.json"

In [ ]:
# log = json.load(open(QUOTA_LOG_PATH, encoding="utf-8")) if os.path.exists(QUOTA_LOG_PATH) else {}
# log[datetime.date.today().isoformat()] = 1   # the 1 real successful call you just made
# json.dump(log, open(QUOTA_LOG_PATH, "w", encoding="utf-8"), indent=2)
# print("Quota log reset for today.")

In [ ]:
def correct_set(name, raw_preds, results_path, daily_quota=DAILY_QUOTA, model_name=MODEL_NAME,
                abort_after_consecutive_failures=5):
    """Resume-safe, with one fix vs. the first version: a failed correction is tagged
    {"used_fallback": True} instead of being saved as if it succeeded, so a rerun
    automatically retries it instead of treating a cached failure as permanently done."""
    today = datetime.date.today().isoformat()
    log = json.load(open(QUOTA_LOG_PATH, encoding="utf-8")) if os.path.exists(QUOTA_LOG_PATH) else {}
    used_today = log.get(today, 0)

    saved = json.load(open(results_path, encoding="utf-8")) if os.path.exists(results_path) else {}
    # discard anything not in the new tagged shape - don't trust old untagged entries
    saved = {fn: v for fn, v in saved.items() if isinstance(v, dict) and "used_fallback" in v}

    done = {fn for fn, v in saved.items() if not v["used_fallback"]}
    remaining = [fn for fn in sorted(raw_preds) if fn not in done]
    print(f"[{name}] {len(done)}/{len(raw_preds)} already genuinely corrected | "
         f"{len(remaining)} remaining | quota used today: {used_today}/{daily_quota}")

    consecutive_failures = 0
    for fn in remaining:
        if used_today >= daily_quota:
            print(f"[{name}] STOPPING: hit today's shared quota ({daily_quota}). Rerun tomorrow.")
            break
        try:
            text = correct_text(raw_preds[fn], model_name=model_name)
            saved[fn] = {"text": text, "used_fallback": False}
            consecutive_failures = 0
        except GeminiCorrectionError as e:
            print(f"  [{name}] {fn} FAILED, keeping raw text for now. detail: {e.detail}")
            saved[fn] = {"text": raw_preds[fn], "used_fallback": True, "error_detail": e.detail}
            consecutive_failures += 1
            if consecutive_failures >= abort_after_consecutive_failures:
                print(f"[{name}] ABORTING: {consecutive_failures} failures in a row - "
                     "something is systemically broken. Fix it before rerunning.")
                json.dump(saved, open(results_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
                json.dump(log, open(QUOTA_LOG_PATH, "w", encoding="utf-8"), indent=2)
                return {fn: v["text"] for fn, v in saved.items()}
        used_today += 1
        log[today] = used_today
        if len(saved) % 10 == 0:
            json.dump(saved, open(results_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            json.dump(log, open(QUOTA_LOG_PATH, "w", encoding="utf-8"), indent=2)
            n_done = sum(1 for v in saved.values() if not v["used_fallback"])
            print(f"  [{name}] progress {n_done}/{len(raw_preds)} genuine | quota today: {used_today}/{daily_quota}")

    json.dump(saved, open(results_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    json.dump(log, open(QUOTA_LOG_PATH, "w", encoding="utf-8"), indent=2)
    n_done = sum(1 for v in saved.values() if not v["used_fallback"])
    print(f"[{name}] finished this run: {n_done}/{len(raw_preds)} genuine | quota today: {used_today}/{daily_quota}")
    return {fn: v["text"] for fn, v in saved.items()}

In [ ]:
# FLEURS: reuse the already-verified raw predictions from EVAL-03
fleurs_raw = json.load(open(RESULTS_PATH, encoding="utf-8"))
print(f"FLEURS raw predictions: {len(fleurs_raw)} (expect 771)")

# Real-world: not previously saved, so transcribe fresh now (only 38 files)
from src.asr import get_asr, transcribe
asr = get_asr(fp16=True)

rw_raw_path = f"{RESULTS_DIR}/finetuned_predictions_real_world.json"
if os.path.exists(rw_raw_path):
    rw_raw = json.load(open(rw_raw_path, encoding="utf-8"))
    print(f"Real-world raw predictions (cached): {len(rw_raw)}")
else:
    rw_raw = {fn: transcribe(os.path.join(REAL_WORLD_DIR, fn), asr) for fn in sorted(real_world_truth)}
    json.dump(rw_raw, open(rw_raw_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"Real-world raw predictions (fresh): {len(rw_raw)}")

In [ ]:
RW_CORRECTED_PATH     = f"{RESULTS_DIR}/corrected_predictions_real_world.json"
FLEURS_CORRECTED_PATH = f"{RESULTS_DIR}/corrected_predictions_fleurs.json"

rw_corrected     = correct_set("real_world", rw_raw, RW_CORRECTED_PATH)
fleurs_corrected = correct_set("fleurs", fleurs_raw, FLEURS_CORRECTED_PATH)

In [ ]:
def eval_pair(ground_truth_dict, raw_preds, corrected_preds, label):
    files = sorted(set(ground_truth_dict) & set(raw_preds) & set(corrected_preds))
    refs = [ground_truth_dict[f] for f in files]
    raw_hyps, corr_hyps = [raw_preds[f] for f in files], [corrected_preds[f] for f in files]

    raw_cer, raw_wer = compute_batch_cer(refs, raw_hyps), compute_batch_wer(refs, raw_hyps)
    corr_cer, corr_wer = compute_batch_cer(refs, corr_hyps), compute_batch_wer(refs, corr_hyps)

    print(f"\n=== {label} (n={len(files)}) ===")
    print(f"Raw fine-tuned   CER: {raw_cer*100:.2f}%   WER: {raw_wer*100:.2f}%")
    print(f"+ LLM corrected  CER: {corr_cer*100:.2f}%   WER: {corr_wer*100:.2f}%")
    print(f"Enhancement Delta: {(raw_cer-corr_cer)*100:.2f} pp CER "
         f"({(raw_cer-corr_cer)/raw_cer*100:.1f}% relative)")
    return dict(label=label, n=len(files), raw_cer=raw_cer, raw_wer=raw_wer, corr_cer=corr_cer, corr_wer=corr_wer)

rw_result     = eval_pair(real_world_truth, rw_raw, rw_corrected, "Real-World")
fleurs_result = eval_pair(ground_truth, fleurs_raw, fleurs_corrected, "FLEURS")

In [ ]:
import random
for fn in random.sample(sorted(real_world_truth), 6):
    print("REFERENCE:", real_world_truth[fn])
    print("RAW:      ", rw_raw[fn])
    print("CORRECTED:", rw_corrected[fn])
    print()

In [ ]:
import pandas as pd

rows = [
    ("FLEURS",     "Baseline",            41.10, 457.15),
    ("FLEURS",     "Fine-tuned",          32.27, 108.11),
    ("FLEURS",     "Fine-tuned + LLM",    31.06, 103.00),
    ("Real-World", "Baseline",            57.89, 576.92),
    ("Real-World", "Fine-tuned",          50.24, 141.96),
    ("Real-World", "Fine-tuned + LLM",    52.22, 139.86),
]
df = pd.DataFrame(rows, columns=["Eval Set", "Pipeline Stage", "CER (%)", "WER (%)"])
print(df.to_string(index=False))
df.to_csv(f"{RESULTS_DIR}/full_pipeline_comparison.csv", index=False)